# 09. 죽은 시간(Dead Time) 분석

## 분석 배경 및 목적

택시 산업에서 **빈차 시간(dead time)**은 운전자가 승객 없이 배회하는 비생산적 시간을 의미하며, 운영 효율성의 핵심 지표다. Cramer & Krueger (2016)는 전통 택시와 Uber를 비교하며, 전통 택시의 빈차율이 Uber 대비 약 40% 높다는 것을 실증적으로 보였다. 이 격차의 주된 원인은 수요-공급의 시공간적 불일치(spatio-temporal mismatch)에 있다.

본 분석은 서울시 택시 운행 데이터(약 6억 건)를 활용하여 다음을 규명한다:

1. **시간대별 빈차율 패턴**: 어느 시간대에 공급과잉/수요과잉이 발생하는가?
2. **공간적 빈차 집중**: 특정 행정동에 빈차 택시가 과도하게 집중되는 현상
3. **재배치 시뮬레이션**: 공급과잉 지역에서 수요과잉 지역으로 택시를 재배치할 때의 매출 증가분 추정

빈차율 = (빈차 시간) / (전체 운행 시간)으로 정의하며, 이 비율이 높을수록 운전자의 시간당 수익이 감소한다. 빈차율의 시공간적 분포를 파악하면 동적 배차(dynamic dispatch) 알고리즘의 설계 근거를 마련할 수 있다.


In [ ]:
import gc, psutil, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows
plt.rcParams['axes.unicode_minus'] = False

CHUNK_SIZE = 1_000_000
D012_PATH = './DC_TBYXD012.csv'

DTYPE_D012 = {
    'PAY_AMT': 'int32',
    'RIDE_DIST': 'float32',
    'VACNTV_DIST': 'float32',
    'RIDE_A_CD': 'category',
    'ALIGHT_A_CD': 'category',
    'RIDE_POS_X': 'float32',
    'RIDE_POS_Y': 'float32',
    'ALIGHT_POS_X': 'float32',
    'ALIGHT_POS_Y': 'float32',
    'DRIVER_ID': 'category',
    'TAXI_VEHC_ID': 'category',
    'PLTF_FEE_AMT': 'int32',
}

def mem_usage():
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

mem_usage()

## 1. 시간대별 시간당 매출 산출 (chunk별 집계)

시간당 매출(revenue per hour)은 택시 운전자의 생산성을 직접적으로 측정하는 지표다. 시간대별로 이를 산출하면 **수요 밀도가 낮아 매출이 급감하는 구간**(dead time)을 객관적으로 식별할 수 있다. Cramer & Krueger (2016)가 제시한 "capacity utilization rate" 개념을 시간대 단위로 세분화한 접근이다.


In [ ]:
# 시간대별 매출 합계, 운행 건수, 총 운행시간(분) 집계
hourly_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['ALIGHT_DTIME'] = chunk['ALIGHT_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    
    # 운행 시간(분)
    ride_dt = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    alight_dt = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk['duration_min'] = (alight_dt - ride_dt).dt.total_seconds() / 60
    chunk['duration_min'] = chunk['duration_min'].clip(lower=0).fillna(0).astype('float32')
    
    grp = chunk.groupby('hour').agg(
        pay_sum=('PAY_AMT', 'sum'),
        trip_count=('PAY_AMT', 'count'),
        duration_sum=('duration_min', 'sum')
    ).reset_index()
    
    hourly_agg = pd.concat([hourly_agg, grp], ignore_index=True)
    
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
        mem_usage()
    del chunk, grp, ride_dt, alight_dt
    gc.collect()

# 최종 집계
hourly_final = hourly_agg.groupby('hour').sum().reset_index()
hourly_final['revenue_per_hour'] = (
    hourly_final['pay_sum'] / (hourly_final['duration_sum'] / 60)
).astype(int)
hourly_final['avg_pay'] = (hourly_final['pay_sum'] / hourly_final['trip_count']).astype(int)

del hourly_agg
gc.collect()

print(hourly_final[['hour', 'trip_count', 'revenue_per_hour', 'avg_pay']])
mem_usage()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#d32f2f' if v <= hourly_final['revenue_per_hour'].quantile(0.2)
          else '#1976d2' for v in hourly_final['revenue_per_hour']]
ax.bar(hourly_final['hour'], hourly_final['revenue_per_hour'], color=colors, edgecolor='white')
ax.set_xlabel('시간대')
ax.set_ylabel('시간당 매출 (원)')
ax.set_title('시간대별 시간당 매출 (하위 20% = 빨간색)')
ax.set_xticks(range(24))
ax.axhline(hourly_final['revenue_per_hour'].median(), ls='--', color='gray', label='중앙값')
ax.legend()
plt.tight_layout()
plt.show()

## 2. 시간대별 빈차율 히트맵

빈차율을 시간대별로 시각화하면, 하루 중 택시 공급이 수요를 초과하는 구간과 부족한 구간을 직관적으로 구분할 수 있다. 히트맵은 다차원 시계열 패턴을 한눈에 파악하기에 적합한 시각화 방식이다.

- **공급과잉 구간**: 빈차율이 높은 시간대 (새벽 2-5시 등)
- **수요과잉 구간**: 빈차율이 낮은 시간대 (출퇴근, 심야 등)


In [ ]:
# 시간대 x 요일별 빈차율 집계
vacancy_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'RIDE_DIST', 'VACNTV_DIST']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    chunk['dow'] = pd.to_datetime(
        chunk['RIDE_DTIME'].str[:8], format='%Y%m%d', errors='coerce'
    ).dt.dayofweek.astype('int8')  # 0=Mon
    
    chunk['total_dist'] = chunk['RIDE_DIST'].fillna(0) + chunk['VACNTV_DIST'].fillna(0)
    
    grp = chunk.groupby(['dow', 'hour']).agg(
        vacntv_sum=('VACNTV_DIST', 'sum'),
        total_sum=('total_dist', 'sum')
    ).reset_index()
    
    vacancy_agg = pd.concat([vacancy_agg, grp], ignore_index=True)
    del chunk, grp
    gc.collect()

vacancy_final = vacancy_agg.groupby(['dow', 'hour']).sum().reset_index()
vacancy_final['vacancy_rate'] = (
    vacancy_final['vacntv_sum'] / vacancy_final['total_sum'].replace(0, np.nan)
).fillna(0)

del vacancy_agg
gc.collect()

pivot_vacancy = vacancy_final.pivot_table(
    index='dow', columns='hour', values='vacancy_rate'
)
dow_labels = ['월', '화', '수', '목', '금', '토', '일']

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot_vacancy, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=range(24), yticklabels=dow_labels, ax=ax)
ax.set_title('요일 x 시간대별 빈차율 (빈차거리 / 총거리)')
ax.set_xlabel('시간대')
ax.set_ylabel('요일')
plt.tight_layout()
plt.show()

del vacancy_final, pivot_vacancy
gc.collect()
mem_usage()

## 3. 시간대 x 행정동별 빈차 배회 패턴 (죽은 시간 탐지)

빈차 배회 패턴을 시간대와 행정동의 2차원으로 교차 분석하면, **특정 지역-시간 조합에서 비효율이 집중되는 hot-spot**을 식별할 수 있다. 이는 택시 운전자의 의사결정(어디서 대기할 것인가)이 실제 수요 분포와 얼마나 괴리가 있는지를 보여준다.


In [ ]:
# 시간대 x 승차 행정동별: 빈차거리 합, 매출 합
dead_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'RIDE_A_CD', 'VACNTV_DIST', 'PAY_AMT']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    chunk['RIDE_A_CD'] = chunk['RIDE_A_CD'].astype(str)
    
    grp = chunk.groupby(['hour', 'RIDE_A_CD']).agg(
        vacntv_sum=('VACNTV_DIST', 'sum'),
        pay_sum=('PAY_AMT', 'sum'),
        trip_count=('PAY_AMT', 'count')
    ).reset_index()
    
    dead_agg = pd.concat([dead_agg, grp], ignore_index=True)
    del chunk, grp
    gc.collect()

dead_final = dead_agg.groupby(['hour', 'RIDE_A_CD']).sum().reset_index()

# 빈차거리 높은데 매출 낮은 구간 = 죽은 시간
dead_final['vacntv_per_trip'] = (
    dead_final['vacntv_sum'] / dead_final['trip_count'].replace(0, np.nan)
).fillna(0).astype('float32')
dead_final['pay_per_trip'] = (
    dead_final['pay_sum'] / dead_final['trip_count'].replace(0, np.nan)
).fillna(0).astype(int)

del dead_agg
gc.collect()

# 죽은 시간 점수: 빈차거리 상위 & 매출 하위
dead_final['vacntv_rank'] = dead_final['vacntv_per_trip'].rank(pct=True)
dead_final['pay_rank'] = dead_final['pay_per_trip'].rank(pct=True)
dead_final['dead_score'] = dead_final['vacntv_rank'] - dead_final['pay_rank']

top_dead = dead_final.nlargest(20, 'dead_score')
print('=== 죽은 시간 Top 20 (빈차 높고 매출 낮은 시간대-행정동) ===')
print(top_dead[['hour', 'RIDE_A_CD', 'trip_count', 'vacntv_per_trip', 'pay_per_trip', 'dead_score']].to_string(index=False))
mem_usage()

In [ ]:
# 죽은 시간 시각화: 시간대별 dead_score 상위 행정동
# 시간대별 평균 dead_score
hour_dead = dead_final.groupby('hour')['dead_score'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(hour_dead['hour'], hour_dead['dead_score'],
       color=['#d32f2f' if s > 0 else '#4caf50' for s in hour_dead['dead_score']],
       edgecolor='white')
ax.set_xlabel('시간대')
ax.set_ylabel('Dead Score (빈차 배회 - 매출)')
ax.set_title('시간대별 죽은 시간 점수 (양수 = 빈차 배회 과다, 매출 부족)')
ax.set_xticks(range(24))
ax.axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.show()

## 4. 죽은 시간 빈차 택시 위치 분포 (행정동별)

죽은 시간으로 식별된 구간에서 빈차 택시의 공간 분포를 행정동 단위로 집계한다. 특정 지역에 빈차가 과도하게 몰리는 현상은 운전자의 **경험 기반 대기 전략(heuristic waiting strategy)**이 비효율적임을 시사한다. 이 분포는 스마트 배차 시스템의 재배치 대상 지역을 선정하는 데 직접적 근거가 된다.


In [ ]:
# 죽은 시간대 식별 (시간당 매출 하위 5개 시간)
dead_hours = hourly_final.nsmallest(5, 'revenue_per_hour')['hour'].tolist()
print(f'죽은 시간대 (시간당 매출 하위 5): {dead_hours}')

# 해당 시간대의 행정동별 빈차 분포
dead_zone_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'RIDE_A_CD', 'VACNTV_DIST', 'TAXI_VEHC_ID']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    chunk = chunk[chunk['hour'].isin(dead_hours)]
    
    if len(chunk) == 0:
        continue
    
    chunk['RIDE_A_CD'] = chunk['RIDE_A_CD'].astype(str)
    grp = chunk.groupby('RIDE_A_CD').agg(
        vacntv_sum=('VACNTV_DIST', 'sum'),
        taxi_count=('TAXI_VEHC_ID', 'nunique')
    ).reset_index()
    
    dead_zone_agg = pd.concat([dead_zone_agg, grp], ignore_index=True)
    del chunk, grp
    gc.collect()

dead_zone_final = dead_zone_agg.groupby('RIDE_A_CD').agg(
    vacntv_sum=('vacntv_sum', 'sum'),
    taxi_count=('taxi_count', 'max')  # 근사치
).reset_index()
dead_zone_final = dead_zone_final.sort_values('vacntv_sum', ascending=False)

del dead_zone_agg
gc.collect()

print(f'\n=== 죽은 시간({dead_hours}) 빈차 집중 행정동 Top 20 ===')
print(dead_zone_final.head(20).to_string(index=False))

In [ ]:
top20 = dead_zone_final.head(20).copy()
top20 = top20.sort_values('vacntv_sum')

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20['RIDE_A_CD'], top20['vacntv_sum'] / 1000,
        color='#e57373', edgecolor='white')
ax.set_xlabel('빈차 거리 합계 (km)')
ax.set_title(f'죽은 시간({dead_hours}시) 빈차 집중 행정동 Top 20')
plt.tight_layout()
plt.show()

del top20
gc.collect()

## 5. 대안 제시: 빈차 재배치 시뮬레이션

행정동별 시간대별 수요(승차 건수) 대비 공급(하차 후 빈차) 갭을 산출한 뒤, 공급 과잉 지역에서 수요 과잉 지역으로 택시를 이동시킬 경우의 매출 증가분을 추정한다.

이 시뮬레이션은 Cramer & Krueger (2016)가 지적한 "전통 택시의 비효율이 정보 비대칭에서 기인한다"는 주장을 서울시 데이터로 검증하는 시도다. 만약 단순 재배치만으로도 유의미한 매출 증가가 추정된다면, 이는 실시간 배차 시스템 도입의 경제적 타당성을 뒷받침한다.


In [ ]:
# 수요: 시간대 x 행정동별 승차건수
# 공급: 시간대 x 행정동별 하차건수 (하차 후 빈차 택시 = 잠재 공급)
demand_agg = pd.DataFrame()
supply_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'ALIGHT_DTIME', 'RIDE_A_CD', 'ALIGHT_A_CD', 'PAY_AMT']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['ALIGHT_DTIME'] = chunk['ALIGHT_DTIME'].astype(str)
    chunk['ride_hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    chunk['alight_hour'] = chunk['ALIGHT_DTIME'].str[8:10].astype('int8')
    chunk['RIDE_A_CD'] = chunk['RIDE_A_CD'].astype(str)
    chunk['ALIGHT_A_CD'] = chunk['ALIGHT_A_CD'].astype(str)
    
    # 수요: 승차 기준
    d = chunk.groupby(['ride_hour', 'RIDE_A_CD']).agg(
        demand=('PAY_AMT', 'count'),
        avg_pay=('PAY_AMT', 'mean')
    ).reset_index()
    d.columns = ['hour', 'area', 'demand', 'avg_pay']
    demand_agg = pd.concat([demand_agg, d], ignore_index=True)
    
    # 공급: 하차 기준 (빈차 발생)
    s = chunk.groupby(['alight_hour', 'ALIGHT_A_CD']).agg(
        supply=('PAY_AMT', 'count')
    ).reset_index()
    s.columns = ['hour', 'area', 'supply']
    supply_agg = pd.concat([supply_agg, s], ignore_index=True)
    
    del chunk, d, s
    gc.collect()

demand_final = demand_agg.groupby(['hour', 'area']).agg(
    demand=('demand', 'sum'),
    avg_pay=('avg_pay', 'mean')
).reset_index()
supply_final = supply_agg.groupby(['hour', 'area']).sum().reset_index()

del demand_agg, supply_agg
gc.collect()

# 수급 갭 산출
gap = demand_final.merge(supply_final, on=['hour', 'area'], how='outer').fillna(0)
gap['gap'] = gap['demand'] - gap['supply']  # 양수=수요과잉, 음수=공급과잉

print('=== 수급 갭 통계 (양수=수요 부족, 음수=공급 과잉) ===')
print(gap.describe())
mem_usage()

In [ ]:
# 죽은 시간대 재배치 시뮬레이션
dead_gap = gap[gap['hour'].isin(dead_hours)].copy()

# 공급 과잉 지역 (빈차 많은 곳)
oversupply = dead_gap[dead_gap['gap'] < 0].nsmallest(10, 'gap')
print('=== 죽은 시간대 공급 과잉 Top 10 (빈차 넘침) ===')
print(oversupply[['hour', 'area', 'demand', 'supply', 'gap']].to_string(index=False))

# 수요 과잉 지역 (택시 부족)
overdemand = dead_gap[dead_gap['gap'] > 0].nlargest(10, 'gap')
print('\n=== 죽은 시간대 수요 과잉 Top 10 (택시 부족) ===')
print(overdemand[['hour', 'area', 'demand', 'supply', 'gap', 'avg_pay']].to_string(index=False))

In [ ]:
# 재배치 시 예상 매출 증가분
# 공급 과잉 지역의 잉여 택시가 수요 과잉 지역으로 이동한다고 가정
total_excess_supply = abs(oversupply['gap'].sum())
total_unmet_demand = overdemand['gap'].sum()

# 이동 가능한 택시 수 = min(잉여 공급, 미충족 수요)
relocatable = int(min(total_excess_supply, total_unmet_demand))

# 수요 과잉 지역의 가중평균 매출
if overdemand['demand'].sum() > 0:
    weighted_avg_pay = (
        (overdemand['avg_pay'] * overdemand['demand']).sum() / overdemand['demand'].sum()
    )
else:
    weighted_avg_pay = 0

expected_revenue = int(relocatable * weighted_avg_pay)

print(f'\n=== 재배치 시뮬레이션 결과 ===')
print(f'공급 과잉 잉여 택시: {total_excess_supply:,.0f} 대/건')
print(f'미충족 수요: {total_unmet_demand:,.0f} 건')
print(f'재배치 가능 건수: {relocatable:,} 건')
print(f'수요 과잉 지역 평균 매출: {weighted_avg_pay:,.0f} 원')
print(f'예상 매출 증가분: {expected_revenue:,} 원')

In [ ]:
# 시각화: 죽은 시간대 수급 갭 Top 지역
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 공급 과잉
os_plot = oversupply.sort_values('gap')
axes[0].barh(os_plot['area'], abs(os_plot['gap']), color='#ef5350', edgecolor='white')
axes[0].set_title(f'공급 과잉 (빈차 넘침) - {dead_hours}시')
axes[0].set_xlabel('잉여 공급 (건)')

# 수요 과잉
od_plot = overdemand.sort_values('gap')
axes[1].barh(od_plot['area'], od_plot['gap'], color='#42a5f5', edgecolor='white')
axes[1].set_title(f'수요 과잉 (택시 부족) - {dead_hours}시')
axes[1].set_xlabel('미충족 수요 (건)')

plt.suptitle('죽은 시간대 택시 수급 불균형', fontsize=14)
plt.tight_layout()
plt.show()

del dead_gap, oversupply, overdemand, os_plot, od_plot
gc.collect()

## 6. 요약

아래 요약은 시간대별 빈차율 패턴, 공간적 빈차 집중 현상, 재배치 시뮬레이션 결과를 종합한다. 실무적으로 이 결과는 (1) 택시 기사 대상 시간대별 운행 가이드라인 제시, (2) 동적 배차 알고리즘의 재배치 우선순위 설정, (3) 공급과잉 지역의 대기 비용 정량화에 활용 가능하다.


In [ ]:
print('=' * 70)
print('죽은 시간 분석 요약')
print('=' * 70)

# 시간당 매출 하위 Top 5
bottom5 = hourly_final.nsmallest(5, 'revenue_per_hour')
print('\n[1] 시간당 매출 하위 시간대 Top 5')
for _, row in bottom5.iterrows():
    print(f'    {int(row["hour"]):02d}시: {row["revenue_per_hour"]:>10,} 원/시간 (건수: {row["trip_count"]:>10,})')

# 해당 시간 빈차 집중 지역
print(f'\n[2] 죽은 시간({dead_hours}시) 빈차 집중 행정동 Top 10')
for _, row in dead_zone_final.head(10).iterrows():
    print(f'    {row["RIDE_A_CD"]}: 빈차거리 {row["vacntv_sum"]/1000:>10,.1f} km')

# 재배치 기대효과
print(f'\n[3] 재배치 시 기대효과')
print(f'    재배치 가능 건수: {relocatable:,} 건')
print(f'    예상 매출 증가분: {expected_revenue:,} 원')
print('=' * 70)

mem_usage()

## 7. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

In [ ]:
# === [시계열 보강] 월별 평균 빈차율 추세 ===
# 죽은 시간(빈차 과다)이 시기적으로 악화/개선되는지 월별로 본다.
_vac = {}
for _ck in _pd.read_csv(D012_PATH, usecols=['RIDE_DTIME','RIDE_DIST','VACNTV_DIST'],
                        dtype={'RIDE_DTIME': str,'RIDE_DIST':'float64','VACNTV_DIST':'float64'},
                        chunksize=1_000_000):
    _ym = _ck['RIDE_DTIME'].str[:6]
    _ck = _ck.assign(ym=_ym)
    _g = _ck.groupby('ym').agg(v=('VACNTV_DIST','sum'),
                               t=('RIDE_DIST', lambda s: (s + _ck.loc[s.index,'VACNTV_DIST']).sum()))
    for _k, _row in _g.iterrows():
        a = _vac.setdefault(_k, [0,0]); a[0]+=_row['v']; a[1]+=_row['t']
    del _ck
_vdf = _pd.DataFrame([(k, v[0]/v[1] if v[1] else 0) for k,v in _vac.items()], columns=['ym','vac_rate'])
_vdf = _vdf[_vdf['ym'].str.match(r'\d{6}')].sort_values('ym')
_vdf['date'] = _pd.to_datetime(_vdf['ym'], format='%Y%m')
fig, ax = _plt.subplots(figsize=(16, 5))
ax.plot(_vdf['date'], _vdf['vac_rate']*100, 'o-', color='#d32f2f', lw=1.5, ms=3)
ax.set_title('월별 평균 빈차율 추세 (빈차거리/총거리)', fontweight='bold')
ax.set_xlabel('월'); ax.set_ylabel('빈차율(%)'); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()

## 7. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

In [ ]:
# === [시계열 보강] 월별 평균 빈차율 추세 ===
# 죽은 시간(빈차 과다)이 시기적으로 악화/개선되는지 월별로 본다.
_vac = {}
for _ck in _pd.read_csv(D012_PATH, usecols=['RIDE_DTIME','RIDE_DIST','VACNTV_DIST'],
                        dtype={'RIDE_DTIME': str,'RIDE_DIST':'float64','VACNTV_DIST':'float64'},
                        chunksize=1_000_000):
    _ym = _ck['RIDE_DTIME'].str[:6]
    _ck = _ck.assign(ym=_ym)
    _g = _ck.groupby('ym').agg(v=('VACNTV_DIST','sum'),
                               t=('RIDE_DIST', lambda s: (s + _ck.loc[s.index,'VACNTV_DIST']).sum()))
    for _k, _row in _g.iterrows():
        a = _vac.setdefault(_k, [0,0]); a[0]+=_row['v']; a[1]+=_row['t']
    del _ck
_vdf = _pd.DataFrame([(k, v[0]/v[1] if v[1] else 0) for k,v in _vac.items()], columns=['ym','vac_rate'])
_vdf = _vdf[_vdf['ym'].str.match(r'\d{6}')].sort_values('ym')
_vdf['date'] = _pd.to_datetime(_vdf['ym'], format='%Y%m')
fig, ax = _plt.subplots(figsize=(16, 5))
ax.plot(_vdf['date'], _vdf['vac_rate']*100, 'o-', color='#d32f2f', lw=1.5, ms=3)
ax.set_title('월별 평균 빈차율 추세 (빈차거리/총거리)', fontweight='bold')
ax.set_xlabel('월'); ax.set_ylabel('빈차율(%)'); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()

---

## References

1. Cramer, J., & Krueger, A. B. (2016). Disruptive Change in the Taxi Business: The Case of Uber. *American Economic Review*, 106(5), 177-182.
2. Yang, H., & Yang, T. (2011). Equilibrium properties of taxi markets with search frictions. *Transportation Research Part B*, 45(4), 696-713.
3. Zhan, X., Qian, X., & Ukkusuri, S. V. (2016). A graph-based approach to measuring the efficiency of an urban taxi service system. *IEEE Transactions on Intelligent Transportation Systems*, 17(9), 2564-2572.
